In [3]:
PROJECT_ROOT

PosixPath('/home/ubuntu/milvus')

In [5]:
import sys
from pathlib import Path

# Adjust this to point to your project root if needed
PROJECT_ROOT = (
    Path(".").resolve().parents[1]
)  # scripts/graph_check -> scripts -> project root


sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
import signal
import multiprocessing as mp
from tqdm import tqdm
from pymilvus import MilvusException
from llama_index.core import Document
from llama_index.core.utils import iter_batch
from llama_index.vector_stores.milvus import MilvusVectorStore
from llama_index.vector_stores.milvus.utils import BM25BuiltInFunction

from src.modules.datasets.feverous.database.feverous_db import FeverousDB
from src.modules.datasets.feverous.utils.feveous_utils import wiki_to_plain_text

from src.modules.datasets.feverous.utils.wiki_page import (
    WikiPage,
    WikiSection,
    WikiTable,
)

In [10]:
DOCUMENT_PATH = "../datas/feverous_wikiv1.db"


In [ ]:
db = FeverousDB('/home/ubuntu/milvus/code/datas/feverous_wikiv1.db')

def read_document(doc_id: str, db: FeverousDB) -> list[Document]:

    page_json = db.get_doc_json(doc_id)
    wiki_page = WikiPage(doc_id, page_json)

    elements = wiki_page.get_page()
    title = wiki_to_plain_text(str(wiki_page.title))

    sections = []
    current_section = f"{title}\n"

    for element in elements:
        if isinstance(element, WikiTable):  # skip table
            continue
        
        if isinstance(element, WikiSection):
            current_section = f"{title}\n"
        
        elif isinstance(element, WikiSection):
            sections.append(current_section)
            current_section = f"{title}\n"
            current_section += f"# {wiki_to_plain_text(str(element))}\n"
        else:
            current_section += f"{wiki_to_plain_text(str(element))} "

    return Document(text=current_section)

In [15]:
doc_ids = db.get_doc_ids()
doc_ids

['',
 '! (Cláudia Pascoal album)',
 '! (The Dismemberment Plan album)',
 '! (The Song Formerly Known As)',
 '! (Trippie Redd album)',
 '! (disambiguation)',
 '!!',
 '!!!',
 '!!! (album)',
 '!!! (disambiguation)',
 '!Action Pact!',
 '!Arriba! La Pachanga',
 '!Hero',
 '!Hero (album)',
 '!Kweiten-ta-ǀǀKen',
 '!T.O.O.H.!',
 '!Women Art Revolution',
 '!Wowow!',
 '" (disambiguation)',
 '"...And Ladies of the Club"',
 '"...The Truth Is a Fucking Lie..."',
 '"900", Cahiers d\'Italie et d\'Europe',
 '"90th Anniversary of the Armed Forces of Azerbaijan (1918–2008)" Medal',
 '"95th Anniversary of the Armed Forces of Azerbaijan (1918–2013)" Medal',
 '"@"',
 '"A" Device',
 '"A" Fort and Battery Hill Redoubt-Camp Early',
 '"A" Is for Alibi',
 '"All God\'s Children" Campaign',
 '"As the Old Sing, So Pipe the Young" (Jan Steen)',
 '"Awaken, My Love!"',
 '"B" Is for Burglar',
 '"Babbacombe" Lee',
 '"Baby Lollipops" murder',
 '"Bad News" Barnes',
 '"Bassy" Bob Brockmann',
 '"Believing Women" in Islam',

In [38]:
for doc_id in tqdm(doc_ids):
    page_json = db.get_doc_json(doc_id)
    wiki_page = WikiPage(doc_id, page_json)

    elements = wiki_page.get_page()
    # title = wiki_to_plain_text(str(wiki_page.title))

    if any(isinstance(element, WikiSection) for element in elements):
        print("Found a WikiSection in the page!")
        break
    # sections = []
    # current_section = f"{title}\n"
    # for element in elements:
    #     if isinstance(element, WikiTable):  # skip table
    #         continue
    #     elif isinstance(element, WikiSection):
    #         sections.append(current_section)
    #         current_section = f"{title}\n"
    #         current_section += f"# {wiki_to_plain_text(str(element))}\n"
    #     else:
    #         current_section += f"{wiki_to_plain_text(str(element))} "
    # sections.append(current_section)  # add the last section

import hashlib


def task(doc_id: str):
    page_json = db.get_doc_json(doc_id)
    wiki_page = WikiPage(doc_id, page_json)

    elements = wiki_page.get_page()
    title = wiki_to_plain_text(str(wiki_page.title))

    sections = []
    current_section = f"{title}\n"
    for element in elements:
        if isinstance(element, WikiTable):  # skip table
            continue

        if isinstance(element, WikiSection):
            sections.append(current_section)
            current_section = f"{title}\n"

        current_section += f"{wiki_to_plain_text(str(element))} "

    sections.append(current_section)  # add the last section

    documents = []
    for section in sections:
        h = hashlib.new("sha256")
        h.update(section.encode())

        document = Document(text=section, doc_id=h.hexdigest())
        documents.append(document)
    return doc_id, documents


task(doc_id)

  0%|          | 1/5421406 [00:00<1:41:38, 889.00it/s]

Found a WikiSection in the page!


('! (Cláudia Pascoal album)',
 [Document(id_='3e04a999e69606073d6865799efd475089cfecddf1134184fcee8c598430c84b', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='! (Cláudia Pascoal album)\n! (pronounced "") is the debut studio album by Portuguese singer Cláudia Pascoal. It was released in Portugal on 27 March 2020 by Universal Music Portugal. The album peaked at number six on the Portuguese Albums Chart. ', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
  Document(id_='b73e272f864a08cd850892f4ebf6dace1dc5ad428c0eb89071e00eef9c7137ef', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', 